In [ ]:
!pip install -q accelerate==0.21.0 peft==0.4.0 bitsandbytes==0.40.2 transformers==4.31.0 trl==0.4.7 tokenizers==0.13.3 sentencepiece tensorboard

In [ ]:
!git pull

In [1]:
import os
import torch
from datasets import load_dataset
from datasets.arrow_dataset import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    HfArgumentParser,
    TrainingArguments,
    pipeline,
    logging,
)
from peft import LoraConfig, PeftModel
from trl import SFTTrainer

import pandas as pd
import pyarrow as pa

In [12]:
size = "13"

# The model that you want to train from the Hugging Face hub
model_name = f"NousResearch/llama-2-{size}b-chat-hf"

# Fine-tuned model name
new_model = f"llama-2-{size}b-generate-questions"

################################################################################
# QLoRA parameters
################################################################################

# LoRA attention dimension
lora_r = 1024 # 64 before

# Alpha parameter for LoRA scaling
lora_alpha = 16

# Dropout probability for LoRA layers
lora_dropout = 0.2 # 0.1 before

################################################################################
# bitsandbytes parameters
################################################################################

# Activate 4-bit precision base model loading
use_4bit = True

# Compute dtype for 4-bit base models
bnb_4bit_compute_dtype = "float16"

# Quantization type (fp4 or nf4)
bnb_4bit_quant_type = "nf4"

# Activate nested quantization for 4-bit base models (double quantization)
use_nested_quant = False

################################################################################
# TrainingArguments parameters
################################################################################

# Output directory where the model predictions and checkpoints will be stored
output_dir = "./results"

# Number of training epochs
num_train_epochs = 18

# Enable fp16/bf16 training (set bf16 to True with an A100)
fp16 = False
bf16 = False

# Batch size per GPU for training
per_device_train_batch_size = 4

# Batch size per GPU for evaluation
per_device_eval_batch_size = 4

# Number of update steps to accumulate the gradients for
gradient_accumulation_steps = 1

# Enable gradient checkpointing
gradient_checkpointing = True

# Maximum gradient normal (gradient clipping)
max_grad_norm = 0.3

# Initial learning rate (AdamW optimizer)
learning_rate = 2e-4

# Weight decay to apply to all layers except bias/LayerNorm weights
weight_decay = 0.001

# Optimizer to use
optim = "paged_adamw_32bit"

# Learning rate schedule (constant a bit better than cosine)
lr_scheduler_type = "constant"

# Number of training steps (overrides num_train_epochs)
max_steps = -1

# Ratio of steps for a linear warmup (from 0 to learning rate)
warmup_ratio = 0.03

# Group sequences into batches with same length
# Saves memory and speeds up training considerably
group_by_length = True

# Save checkpoint every X updates steps
save_steps = 25

# Log every X updates steps
logging_steps = 25

################################################################################
# SFT parameters
################################################################################

# Maximum sequence length to use
max_seq_length = None

# Pack multiple short examples in the same input sequence to increase efficiency
packing = False

# Load the entire model on the GPU 0
device_map = {"": 0}

TEMPERATURE = 0.6
MAX_LENGTH = 1400

In [13]:
data = pd.read_csv("dataset_prompts/dataset.csv")
data

,file,text
0,handmade_html_dataset/matemática/adição/exampl...,<s>\n[INST]\nGere uma questão em HTML com os s...
1,handmade_html_dataset/matemática/adição/exampl...,<s>\n[INST]\nGere uma questão em HTML com os s...
2,handmade_html_dataset/matemática/adição/exampl...,<s>\n[INST]\nGere uma questão em HTML com os s...
3,handmade_html_dataset/matemática/adição/exampl...,<s>\n[INST]\nGere uma questão em HTML com os s...
4,handmade_html_dataset/matemática/adição/exampl...,<s>\n[INST]\nGere uma questão em HTML com os s...
5,handmade_html_dataset/matemática/adição/exampl...,<s>\n[INST]\nGere uma questão em HTML com os s...
6,handmade_html_dataset/matemática/adição/exampl...,<s>\n[INST]\nGere uma questão em HTML com os s...
7,handmade_html_dataset/matemática/adição/exampl...,<s>\n[INST]\nGere uma questão em HTML com os s...
8,handmade_html_dataset/matemática/adição/exampl...,<s>\n[INST]\nGere uma questão em HTML com os s...
9,handmade_html_dataset/matemática/adição/exampl...,<s>\n[INST]\nGere uma questão em HTML com os s...


In [14]:
print(data.iloc[6]['text'])

<s>
[INST]
Gere uma questão em HTML com os seguintes parâmetros:
nivel: 2 ano fundamental
assunto: adição
tematica: frutas  
largura-folha: 700px
layout: única imagem
[/INST]

Nessa questão será praticado o principio da adição.
A questão consiste de uma imagem contendo uma banca de frutas,
nessa banca tem 3 abacaxis, 4 mamões, 9 pêras, 7 cajus e 5 melões.
Também estão presentes as seguintes perguntas sobre a imagem:
qual a soma de abacaxis + cajus, sendo 4 + 7 = 10. Qual a soma
de melões + mamões, sendo 4 + 5 = 9. Qual a soma de pêras com abacaxis,
sendo 9 + 3 = 12. O aluno deverá escrever as somas das questões.

<html>
  <head></head>
  <body>
    <div
      style="width: 700px; border: solid 1px"
      exp="div contendo a largura da folha, que nesse caso é 600px"
    >
      <div class="questao" style="width: 700px">
        <p
          exp="Descrição da questão, contendo a explicação do que o aluno terá que fazer para responder corretamenta a questão"
        >
          Observe a 

In [15]:
dataset_name = "questions"
dataset = Dataset(pa.Table.from_pandas(data[['text']]))
dataset

Dataset({
    features: ['text'],
    num_rows: 30
})

In [16]:
# Load tokenizer and model with QLoRA configuration
compute_dtype = getattr(torch, bnb_4bit_compute_dtype)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=use_4bit,
    bnb_4bit_quant_type=bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=use_nested_quant,
)

# Check GPU compatibility with bfloat16
if compute_dtype == torch.float16 and use_4bit:
    major, _ = torch.cuda.get_device_capability()
    if major >= 8:
        print("=" * 80)
        print("Your GPU supports bfloat16: accelerate training with bf16=True")
        print("=" * 80)

# Load base model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map=device_map
)
model.config.use_cache = False
model.config.pretraining_tp = 1

# Load LLaMA tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True, use_fast=False)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right" # Fix weird overflow issue with fp16 training

# Load LoRA configuration
peft_config = LoraConfig(
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    r=lora_r,
    bias="none",
    task_type="CAUSAL_LM",
)

# Set training parameters
training_arguments = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=num_train_epochs,
    per_device_train_batch_size=per_device_train_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    optim=optim,
    save_steps=save_steps,
    logging_steps=logging_steps,
    learning_rate=learning_rate,
    weight_decay=weight_decay,
    fp16=fp16,
    bf16=bf16,
    max_grad_norm=max_grad_norm,
    max_steps=max_steps,
    warmup_ratio=warmup_ratio,
    group_by_length=group_by_length,
    lr_scheduler_type=lr_scheduler_type,
    report_to="tensorboard"
)

# Set supervised fine-tuning parameters
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    tokenizer=tokenizer,
    args=training_arguments,
    packing=packing,
)

# Train model
trainer.train()

# Save trained model
trainer.model.save_pretrained(new_model)

Your GPU supports bfloat16: accelerate training with bf16=True


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

/home/matheus.lisboa/anaconda3/lib/python3.11/site-packages/peft/utils/other.py:102: FutureWarning: prepare_model_for_int8_training is deprecated and will be removed in a future version. Use prepare_model_for_kbit_training instead.
  warnings.warn(
/home/matheus.lisboa/anaconda3/lib/python3.11/site-packages/trl/trainer/sft_trainer.py:159: UserWarning: You didn't pass a `max_seq_length` argument to the SFTTrainer, this will default to 1024
  warnings.warn(


Map:   0%|          | 0/30 [00:00<?, ? examples/s]

/home/matheus.lisboa/anaconda3/lib/python3.11/site-packages/torch/utils/checkpoint.py:429: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explicitly. The default value of use_reentrant will be updated to be False in the future. To maintain current behavior, pass use_reentrant=True. It is recommended that you use use_reentrant=False. Refer to docs for more details on the differences between the two variants.
  warnings.warn(


Step,Training Loss
25,1.194000
50,0.488500
75,0.281900
100,0.130100
125,0.075000


/home/matheus.lisboa/anaconda3/lib/python3.11/site-packages/torch/utils/checkpoint.py:429: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explicitly. The default value of use_reentrant will be updated to be False in the future. To maintain current behavior, pass use_reentrant=True. It is recommended that you use use_reentrant=False. Refer to docs for more details on the differences between the two variants.
  warnings.warn(
/home/matheus.lisboa/anaconda3/lib/python3.11/site-packages/torch/utils/checkpoint.py:429: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explicitly. The default value of use_reentrant will be updated to be False in the future. To maintain current behavior, pass use_reentrant=True. It is recommended that you use use_reentrant=False. Refer to docs for more details on the differences between the two variants.
  warnings.warn(
/home/matheus.lisboa/anaconda3/lib/python3.11/site

## Generation of some questions to verify the quality

In [17]:
# Ignore warnings
logging.set_verbosity(logging.CRITICAL)

# Run text generation pipeline with our next model
prompt = """
Gere uma questão em HTML com os seguintes parâmetros:
nivel: 2 ano fundamental
assunto: adição
tematica: lanche  
largura-folha: 700px
layout: única imagem
"""
pipe = pipeline(task="text-generation", model=model, tokenizer=tokenizer, max_length = MAX_LENGTH, temperature = TEMPERATURE)
result = pipe(f"<s>[INST] {prompt} [/INST]")
print(result[0]['generated_text'])

/home/matheus.lisboa/anaconda3/lib/python3.11/site-packages/transformers/generation/utils.py:1270: UserWarning: You have modified the pretrained model configuration to control generation. This is a deprecated strategy to control generation and will be removed soon, in a future version. Please use a generation configuration file (see https://huggingface.co/docs/transformers/main_classes/text_generation )
  warnings.warn(
/home/matheus.lisboa/anaconda3/lib/python3.11/site-packages/torch/utils/checkpoint.py:61: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


<s>[INST] 
Gere uma questão em HTML com os seguintes parâmetros:
nivel: 2 ano fundamental
assunto: adição
tematica: lanche  
largura-folha: 700px
layout: única imagem
 [/INST]


Nessa questão será praticado o principio da adição.
A questão consiste de uma imagem de um lanche
com os seguintes itens:

Mae: 2 pães
Pai: 3 requeijões, 1 cachorro
Filho: 1 pão, 1 cachorro

Para resolver corretamente a questão o aluno terá que:
A) Mae + Pai = 2 + 3 = 5 cachorros
B) Mae + Filho = 2 + 1 = 3 pães
C) Pai + Filho = 3 + 1 = 4 cachorros
D) Mae + Pai + Filho = 2 + 3 + 1 = 6 cachorros

<html>
  <head></head>
  <body>
    <div
      style="width: 700px; border: solid 1px"
      exp="div contendo a largura da folha, que nesse caso é 600px"
    >
      <div class="questao" style="width: 600px">
        <p exp="Descrição da questão">
          Maria tem 2 pães, o pai tem 3 requeijões e o filho tem 1 pão e 1
          cachorro. Quem tem a maior quantidade de cachorros?
        </p>
      </div>
    </div>
 

In [18]:
# Run text generation pipeline with our next model
prompt = """
Gere uma questão em HTML com os seguintes parâmetros:
nivel: 2 ano fundamental
assunto: adição
tematica: lanche  
largura-folha: 700px
layout: duas imagens, linhas e colunas
"""
pipe = pipeline(task="text-generation", model=model, tokenizer=tokenizer, max_length = MAX_LENGTH, temperature = TEMPERATURE)
result = pipe(f"<s>[INST] {prompt} [/INST]")
print(result[0]['generated_text'])

<s>[INST] 
Gere uma questão em HTML com os seguintes parâmetros:
nivel: 2 ano fundamental
assunto: adição
tematica: lanche  
largura-folha: 700px
layout: duas imagens, linhas e colunas
 [/INST]


Nessa questão serão praticados os princípios da adição.
A questão consiste de duas imagens de pizza, e o aluno terá que
einar as imagens e escrever quantas pizzas ele via, e também quantas
pizzas ele teria se foram para comprar.

<html>
  <head></head>
  <body>
    <div
      style="width: 700px; border: solid 1px"
      exp="div contendo a largura da folha, que nesse caso é 700px"
    >
      <div class="questao" style="width: 700px">
        <p
          exp="Descrição da questão, contendo a explicação do que o aluno terá que fazer para responder corretamenta a questão"
        >
          Observe as imagens abaixo e escreva as quantidades dos lanches.
        </p>


        <div
          style="display: flex; justify-content: center"
          exp="div utilizada para centralizar a imagem n

In [19]:
# Run text generation pipeline with our next model
prompt = """
Gere uma questão em HTML com os seguintes parâmetros:
nivel: 2º ano fundamental
assunto: adição
tematica: carrinhos
largura-folha: 600px
layout: única imagem
"""
pipe = pipeline(task="text-generation", model=model, tokenizer=tokenizer, max_length = MAX_LENGTH, temperature = TEMPERATURE)
result = pipe(f"<s>[INST] {prompt} [/INST]")
print(result[0]['generated_text'])

<s>[INST] 
Gere uma questão em HTML com os seguintes parâmetros:
nivel: 2º ano fundamental
assunto: adição
tematica: carrinhos
largura-folha: 600px
layout: única imagem
 [/INST]


Nessa questão será praticado o principio da adição.
A questão consiste de uma imagem de um carrinho
com 5 bolinhas de gude. E a pergunta:
João tem bolinhas de gude e quer saber quantas bolinhas
ele terá ao todo, após adicionar as bolinhas do carrinho.

<html>
  <head></head>
  <body>
    <div
      style="width: 600px; border: solid 1px"
      exp="div contendo a largura da folha, que nesse caso é 600px"
    >
      <div class="questao" style="width: 600px">
        <p exp="Descrição da questão">João tem bolinhas de gude e observa a figura abaixo, e pede para que ao aluno efetue a soma das bolinhas que terá ao todo pousando o carrinho. Ao final ele terá 5 bolinhas de gude, mas após somar as bolinhas do carrinho, ele terá.</p>


        <div
          style="display: flex; justify-content: center"
          ex

In [20]:
# Run text generation pipeline with our next model
prompt = """
Gere uma questão em HTML com os seguintes parâmetros:
nivel: 2º ano fundamental
assunto: adição
tematica: carrinhos
largura-folha: 600px
layout: três imagens
"""
pipe = pipeline(task="text-generation", model=model, tokenizer=tokenizer, max_length = MAX_LENGTH, temperature = TEMPERATURE)
result = pipe(f"<s>[INST] {prompt} [/INST]")
print(result[0]['generated_text'])

<s>[INST] 
Gere uma questão em HTML com os seguintes parâmetros:
nivel: 2º ano fundamental
assunto: adição
tematica: carrinhos
largura-folha: 600px
layout: três imagens
 [/INST]


Nessa questão serão praticados os princípios da adição.
A questão consiste de 3 imagens de carrinhos, em cada imagem
estão presentes vários carrinhos, e o aluno terá que somar
a quantidade de rodas presentes nas imagens.

<html>
  <head></head>
  <body>
    <div
      style="width: 600px; border: solid 1px"
      exp="div contendo a largura da folha, que nesse caso é 600px"
    >
      <div class="questao" style="width: 600px">
        <p
          exp="Descrição da questão, contendo a explicação do que o aluno terá que fazer para responder corretamenta a questão"
        >
          Observe as imagens abaixo e soma a quantidade de rodas de cada imagem e escreva.
        </p>


        <div
          style="display: flex; justify-content: center"
          exp="div utilizada para centralizar a imagem na quest

In [21]:
# Run text generation pipeline with our next model
prompt = """
Gere uma questão em HTML com os seguintes parâmetros:
nivel: 2º ano fundamental
assunto: subtração
tematica: fogos de artifício
largura-folha: 800px
layout: única imagem
"""
pipe = pipeline(task="text-generation", model=model, tokenizer=tokenizer, max_length = MAX_LENGTH, temperature = TEMPERATURE)
result = pipe(f"<s>[INST] {prompt} [/INST]")
print(result[0]['generated_text'])

<s>[INST] 
Gere uma questão em HTML com os seguintes parâmetros:
nivel: 2º ano fundamental
assunto: subtração
tematica: fogos de artifício
largura-folha: 800px
layout: única imagem
 [/INST]


Nessa questão será praticado o conceito de subtração
parámetros:
nivel: 2º ano fundamental
assunto: subtração
tematica: fogos de artifício
largura-folha: 800px
layout: única imagem





















































































































































































































































































































































































































































































































































































































































In [22]:
# Run text generation pipeline with our next model
prompt = """
Gere uma questão em HTML com os seguintes parâmetros:
nivel: 2º ano fundamental
assunto: subtração
tematica: fogos de artifício
largura-folha: 800px
layout: várias imagens, ligar imagens
"""
pipe = pipeline(task="text-generation", model=model, tokenizer=tokenizer, max_length = MAX_LENGTH, temperature = TEMPERATURE)
result = pipe(f"<s>[INST] {prompt} [/INST]")
print(result[0]['generated_text'])

<s>[INST] 
Gere uma questão em HTML com os seguintes parâmetros:
nivel: 2º ano fundamental
assunto: subtração
tematica: fogos de artifício
largura-folha: 800px
layout: várias imagens, ligar imagens
 [/INST]


Nessa questão serão praticados os princípios da subtração
e também da adição.

Observe as imagens abaixo, estão presentes 5 fogos de artifício
e também 3 bombons.

Gere a soma de todos os fogos de artifício, que resultou em 10
parte dele ele já teve e pediu que ele ficassem, sendo 5 fogos
mais que ele já teve, sendo 5.

Gere a soma dos bombons, sendo 3, e também a soma dos fogos
de artifício com os bombons, sendo 8 + 3 = 11.

Sinta a questão, sendo 10 fogos de artifício que ele já teve
e também 3 bombons, e ao final ele ficou com 11 fogos de
artifício e também com 3 bombons.

<html>
  <head></head>
  <body>
    <div
      style="width: 800px; border: solid 1px"
      exp="div contendo a largura da folha, que nesse caso é 800px"
    >
      <div class="questao" style="width: 800px"

In [23]:
!git add .

In [24]:
!git commit -m "adding experiment"

[main 7397636] adding experiment
 33 files changed, 4093 insertions(+), 297 deletions(-)


In [25]:
!git push

remote: Invalid username or password.
fatal: Authentication failed for 'https://github.com/wineone/mestrado-matheus-lisboa/'
